**Ingest Results.csv file**
1. Read file using spark dataframe reader API
2.add Metadata Columns 
-   source file
- ingestion Timestamp
3. write to bonze delta table

In [0]:
%run ../00.Common/01.environment-config


In [0]:
%run ../00.Common/02.bronze-helpers


In [0]:
source_file = f"{landing_folder_path}/results"
table_name = f"{catalog_nameone}.{bronze_schema}.results"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType, IntegerType, FloatType


result_schema = StructType([
    StructField("date", DateType()),
    StructField("raceName", StringType()),
    StructField("round", IntegerType()),
    StructField("season", StringType()),
    StructField("url", StringType()),
    StructField("constructorId",StringType()),
    StructField("driverId", StringType()),
    StructField("grid", IntegerType()),
    StructField("laps", IntegerType()),
    StructField("number", IntegerType()),
    StructField("points", FloatType()),
    StructField("position", IntegerType()),
    StructField("positionText", StringType()),
    StructField("status", StringType()),

])


results_df = (
    spark.read.format("json")
    .schema(result_schema)
    .option("mode", "FAILFAST")
    .load(source_file)
)

results_final_df = add_ingestion_metadata(results_df)
display(results_final_df)

In [0]:
results_final_df.write.format('delta').mode('overwrite').saveAsTable(table_name)